# Structured Model Output with Pydantic

**Course author:** Jayneel Girnara

Learn how to turn free-form model responses into validated Python objects with Pydantic schemas.

## Learning goals
- See why plain text extraction can be inconsistent.
- Define a schema with typed fields and descriptions.
- Bind a schema to a chat model and access validated fields.
- Reuse schemas for more complex output such as a list of movies.

## Before you start
Install the dependencies and configure `OPENAI_API_KEY` in a local .env file. Never put a real key in a notebook or commit .env. The text below is example input; replace personal details with synthetic values before sharing.

## How to work through it
Run each cell in order. Compare the unstructured extraction with the schema-based response, then inspect the field constraints in the second example. API calls may incur charges.

## 1. Extract fields with a free-form response

A plain-text instruction asks the model to extract fields, but the returned content is still just text. Its wording or formatting may vary, and code would have to parse and validate it separately.

**Checkpoint:** Does asking for an email address make the response a typed Python value?

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-5")

In [10]:
text="Hello my name is Alex Example,Email is alex@example.com and my age is 19"

In [ ]:
res=llm.invoke(f"Extract the name, email address, and age from this text:{text}")
print(type(res.content))

## 2. Produce validated structured output

Declare the expected fields once, bind the schema to the model, and access the validated result as attributes.

A **schema** describes the expected shape of data. A Pydantic `BaseModel` declares fields and Python types; `Field` can add constraints or descriptions. `with_structured_output` asks the supported model integration for that shape, and Pydantic parses the returned data. Accessing `res.email` reads a validated field. Validation checks structure and types, not whether the model extracted a true fact.

**Checkpoint:** What kind of error can a schema catch, and what kind of error can it not catch?

## 2. Validate fields with a Pydantic schema

## 3. Inspect the validated fields

The model instance exposes its fields as attributes. Required fields must be present; declared types and constraints are validated when the result is parsed.

In [23]:
from pydantic import BaseModel,Field
from typing import List

class ResponseStructure(BaseModel):
    name:str = Field(description="Complete name")
    email:str = Field(description="Email Address")
    age:int = Field(description="Age")


structured_llm=llm.with_structured_output(ResponseStructure)

In [ ]:
res=structured_llm.invoke(f"Extract the name, email address, and age from this text:{text}")


In [19]:
res.email

'alex@example.com'

## 3. Model a richer structured response

A nested schema can represent repeated records: `Movies` describes one movie, while `AllMovies` holds a list of them. The response is therefore a structured object containing multiple validated items, rather than an unstructured paragraph.

**Checkpoint:** Which model represents one item, and which model represents the collection?

In [24]:
class Movies(BaseModel):
    title:str = Field(description="Movie Title")
    year:int = Field(description="Moive Released Year")

class AllMovies(BaseModel):
    movies:List[Movies]
    

movie_llm_structured=llm.with_structured_output(AllMovies)

In [25]:
movie_llm_structured.invoke("Give me 3 Scifi movie")

/Users/Alex/Desktop/GENAI_Series/env/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AllMovies(movies=[Movies(...e Odyssey', year=1968)]), input_type=AllMovies])
  return self.__pydantic_serializer__.to_python(


AllMovies(movies=[Movies(title='The Matrix', year=1999), Movies(title='Blade Runner', year=1982), Movies(title='2001: A Space Odyssey', year=1968)])